In [41]:
from dotenv import load_dotenv, find_dotenv
from langchain.chat_models import init_chat_model
from pprint import pprint
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver  
from langchain.messages import HumanMessage, AIMessage, SystemMessage
from tavily import TavilyClient
import os
from typing import Dict, Any
from pydantic import BaseModel

In [9]:
load_dotenv(find_dotenv(), override=True)

True

In [10]:
model = init_chat_model(
                        model="gpt-5-nano", 
                        use_responses_api=True
                        )

In [65]:
# system_prompt = """
# You are LeftoverChef, a creative and practical home cooking assistant.

# Your purpose is to help users turn leftover ingredients into delicious meals while minimizing food waste.

# You have web search tool to search the web for recipes that can be made with the ingredients provided by the user.

# Rules:
# 1. Always prioritize using the ingredients provided by the user.
# 2. Suggest recipes that make maximum use of leftovers before recommending additional ingredients.
# 3. Assume the user is a home cook with basic kitchen equipment unless otherwise specified.
# 4. If important information is missing (such as available protein, vegetables, dietary restrictions, or cooking appliances), ask concise follow-up questions.
# 5. Recommend realistic recipes that can be prepared with common cooking techniques.
# 6. If additional ingredients would significantly improve the dish, clearly separate:
#    - Ingredients already available
#    - Optional ingredients to purchase
# 7. When multiple dishes are possible, provide the best one based on:
#    - Best use of leftovers
#    - Easiest to make
#    - Most flavorful/creative
# 8. Consider dietary requirements if provided.
# 9. Never waste usable ingredients without explanation.
# 10. Be encouraging and help users gain confidence in cooking.

# For every recipe suggestion, use the following format:

# DISH NAME
# Why this works:
# - Brief explanation

# Uses These Leftovers:
# - List ingredients

# Additional Ingredients Needed:
# - List (or "None")

# Instructions:
# 1. Step 1
# 2. Step 2
# 3. Step 3

# Estimated Time:
# - X minutes

# Difficulty:
# - Easy / Medium / Hard

# Leftover Utilization Score:
# - X/10

# If ingredients seem unusual or insufficient, suggest creative combinations rather than immediately asking the user to buy more food.

# Your goal is to reduce food waste, save money, and create tasty meals from what the user already has.
# """

system_prompt = """You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

When multiple dishes are possible, provide the top three based on:
- Best use of leftovers
- Easiest to make
- Most flavorful/creative
"""

In [25]:
# web search tool

tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

@tool
def web_search_tool(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

In [67]:
# Structured output schema for recipe suggestions

class Recipe(BaseModel):
    dish_name: str
    why_this_works: str
    ingredients_used: list[str]
    instructions: list[str]
    estimated_time: str
    difficulty: str

In [73]:
# initialising agent with memory and giving it web search tool to use for looking up information and configuring it 
# to use a specific thread_id for the conversation so that it can remember the context of the conversation and use 
# it in future interactions with the user.

memory = InMemorySaver()

agent = create_agent(
    model = model,
    system_prompt=system_prompt,
    # response_format=Recipe,
    tools = [web_search_tool],
    checkpointer=memory,  
)

config = {
    "configurable": {
        "thread_id": "user-1"
    }
}

Sample input:

- I have leftover rice, chicken, and carrots. What can I make?
- I have half an onion, 2 boiled potatoes, and some spinach.
- I need dinner ideas using leftover turkey and mashed potatoes.

Inputs with constraints:

- I have leftover rice, eggs, and peas. I only have 15 minutes to cook.
- I have tortillas, beans, and leftover steak. Give me a Mexican-style dish.
- I'm vegetarian. I have rice, chickpeas, and broccoli.

In [75]:
user_input = input("\nInput: ")

response = agent.invoke(
    {"messages": [HumanMessage(content=user_input)]},
    config=config,
)

print(response['messages'][-1].text)

Nice—rice, chickpeas, and broccoli are a great base for vegetarian meals. Here are the top three options that best use leftovers, are easy to make, and offer good flavor/creativity.

1) Cheesy Broccoli Chickpea & Rice Casserole (great for leftovers)
- Why it fits: Built for batch cooking; easy to reheat; uses all three ingredients.
- What you’ll make: Rice, chickpeas, and broccoli baked with a cheese sauce.
- Link: https://clickncook.org/recipe/one-pot-cheesy-broccoli-chickpea-rice-casserole

2) Broccoli, Chickpea and Brown Rice Bowl with Mustard-Soy Dressing (easy weeknight)
- Why it fits: Quick, minimal hands-on time; super simple assembly; great for weeknights.
- What you’ll make: Brown rice, broccoli, and chickpeas with a Dijon–soy dressing.
- Link: https://www.rachaelhartleynutrition.com/blog/2014/01/broccoli-chickpea-and-brown-rice-bowl

3) Vegan Teriyaki Chickpea Bowl (flavorful and creative)
- Why it fits: A bold, tasty option that feels a bit special without extra work.
- What

In [69]:
user_input = input("\nInput: ")

result = agent.invoke(
    {"messages": [HumanMessage(content=user_input)]},
    config=config,
)

recipe = result["structured_response"]
print(recipe)

dish_name='One-Pot Cheesy Broccoli Chickpea Rice Casserole' why_this_works='Best use of leftovers: this one-pot casserole brings together rice, chickpeas, and broccoli in a comforting, cohesive dish with minimal cleanup. It’s easy to scale up or down and can be tucked into a weeknight meal plan.' ingredients_used=['1 cup cooked brown rice (or white rice)', '1 can chickpeas, drained', '2 cups broccoli florets', '1 small onion, diced', '2 cloves garlic, minced', '1 cup shredded cheese (cheddar or your choice)', '1–1.5 cups vegetable broth (or water)', '1 tablespoon olive oil', 'Salt and pepper to taste', 'Optional: 1/2 teaspoon paprika or chili flakes for a hint of heat', 'Optional: parsley or chives for topping'] instructions=['1. Preheat oven to 375°F (190°C) if finishing in the oven; otherwise you can finish on the stovetop for a lighter version.', '2. In a large skillet, heat olive oil over medium heat. Sauté onion until translucent (3–4 minutes). Add garlic and cook 30 seconds more.

In [70]:
print(f"""
Dish Name:
{recipe.dish_name}

Why This Works:
{recipe.why_this_works}

Ingredients Used:
{", ".join(recipe.ingredients_used)}

Estimated Time:
{recipe.estimated_time}

Difficulty:
{recipe.difficulty}
""")


Dish Name:
One-Pot Cheesy Broccoli Chickpea Rice Casserole

Why This Works:
Best use of leftovers: this one-pot casserole brings together rice, chickpeas, and broccoli in a comforting, cohesive dish with minimal cleanup. It’s easy to scale up or down and can be tucked into a weeknight meal plan.

Ingredients Used:
1 cup cooked brown rice (or white rice), 1 can chickpeas, drained, 2 cups broccoli florets, 1 small onion, diced, 2 cloves garlic, minced, 1 cup shredded cheese (cheddar or your choice), 1–1.5 cups vegetable broth (or water), 1 tablespoon olive oil, Salt and pepper to taste, Optional: 1/2 teaspoon paprika or chili flakes for a hint of heat, Optional: parsley or chives for topping

Estimated Time:
45–60 minutes total ( stovetop version is shorter, about 25–35 minutes)

Difficulty:
Easy



In [52]:
def display_recipe(recipe):
    print("\n" + "=" * 100)
    print(f"🍽️  {recipe.dish_name}")
    print("=" * 100)

    print(f"\nWhy This Works:\n{recipe.why_this_works}")

    print("\nIngredients Used:")
    for ingredient in recipe.ingredients_used:
        print(f"  • {ingredient}")

    print(f"\nEstimated Time: {recipe.estimated_time}")
    print(f"Difficulty: {recipe.difficulty}")

    if recipe.instructions:
        print("\nInstructions:")
        for i, step in enumerate(recipe.instructions, start=1):
            print(f"  {i}. {step}")

display_recipe(recipe)


🍽️  What to make with leftover rice, chicken, and carrots ( recipe ideas )

Why This Works:
These options use your exact ingredients and keep prep simple, from quick stovetop pans to easy oven bakes.

Ingredients Used:
  • leftover rice
  • chicken
  • carrots

Estimated Time: Approx. 20–60 minutes depending on option
Difficulty: Easy to Easy-Moderate


In [55]:
# confirm if tool calls were made

state = agent.get_state(config)
pprint(state.values)

{'messages': [HumanMessage(content='I have leftover rice, chicken, and carrots. What can I make?', additional_kwargs={}, response_metadata={}, id='30b6213e-20cb-41bc-8804-81caa81f6b34'),
              AIMessage(content=[{'id': 'rs_0159d51a0956678c006affd9b262b887d081e11cf760d6dd9a', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABq_9m1SPxeYE8yLV1Sxyttao_iPlFgNaFbl5WpR4u1eIQQjbRYmeHmAkaNPc9NPu03jXbStephl4bdRaObBQ8Ovp96BL5fLonMNGRwGKqoDpDzk36GcLFaBFDya9JqJEE7QkVzyXb9FG2T7aej2GeE1HO1tRjRFoSfJy5yQ5xDMnc6OL19vUlEZWRexLVxcgN_h2ChmXuaVswyYg4jcVja22_i0RDQW96SKGU6DnJnp26h0r5k1wFbZJD40CfbVqYDWyAkwDIzCeoiETfI2udV0kLH8cKgyW8cVkXvPoTQtA6fhmEy1LF50YutdjDSai1xmzK7YN3vxyiL_Wi_r18T_cSXMFC4OrXBKtKzPUS8Dkf2w-2VyukqKAtvpnPP-ZD3ncsPIIDQQJ3B0I15jc4sR-IlYkzfHNdOBI4avIS4HoZoa0mA4XjPaVW-qHoimHIz6v7xLqVgFnF1PAp-25q3V2Q2PD2N02THpqjTarf1UnQATbvathufBYxGICjQ2A6oThPAqh3jYBp5Uy3qEJ6lXEEqPIF84Z67T4aGVTP_hHWLRMUXszAqP0hvbeWeKTGWwSYnlwCZbDYRptIUxaOkAxgEWJhXrZzXqV5RHC3HGk8D0wxkvThxsn5zjba